# N116 · 轮廓DP与窄网格状态

**目标：** 仅保存已处理区域与未来相交的边界信息。

**先修：** N112, N087, N128。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 行掩码；合法状态；相邻行兼容；多进制状态；复杂度按宽度增长。

**配套讲解来源：** [同名逐章意见](../../comment/116_profile_dp_grid_states.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

网格题的行数可以很大，但**宽度很窄**时，我们不需要记住整张网格，只要记住"已处理部分与未处理部分的交界"——也就是上一整行的状态。这就是轮廓 DP（逐行 DP）。本章三个具体问题：

1. **考场排座（LC1349）：** 教室座位有 '.'（能坐）和 '#'（坏了）。学生不能坐在别人的左右相邻、左上/右上/左下/右下斜对角位置，但坐在正前后是可以的。问最多能坐多少人。
2. **三色染网格（LC1931）：** 用 3 种颜色染 m×n 网格，左右相邻、上下相邻的格子颜色都必须不同，问有多少种染法（答案对 10^9+7 取模）。
3. **N×3 网格染色计数（LC1411）：** 上一个问题在宽度固定为 3 时的快速版本，用两个数字就能递推出答案。

**具体到数字的例子：** 座位表 `['.#.','...']`（两行三列，第一行中间坏了），最多坐 **4** 人。为什么？第一行只能坐第 0、2 列（第 1 列是坏座），共 2 人；第二行如果有人坐第 1 列，会同时和第一行第 0、2 列的人斜对角冲突，所以第二行也只能坐第 0、2 列，共 2 人；合计 4。再比如 `num_of_ways_n3(1)`：单独一行三格，染法有 12 种（每种"首尾同色"型 6 种 + "三格互不相同"型 6 种）。

## 2. 基础知识：先读懂这些词

- **轮廓DP（逐行DP）：** 一行一行地推进 DP，状态只保留"刚处理完的那一行"。因为规则只涉及相邻行，更早的行对未来的影响已经全部体现在这一行里，可以忘掉。它省内存的关键就在"遗忘"。
- **行掩码（row mask）：** 把一行的选择压成一个二进制数，第 i 位是 1 表示第 i 列做了某个动作（坐人/选了某颜色）。一行 W 列就是 2^W 种掩码——**指数开销放在宽度 W 上，而不是整个面积上**，所以宽度必须小。
- **合法状态（行内约束预过滤）：** 先把违反行内规则的掩码筛掉（比如坐人不能左右相邻），DP 只在合法状态之间转移。预先过滤比在转移里判断更省事。
- **相邻行兼容（compatibility）：** 上一行的掩码和这一行的掩码放在一起不违反跨行规则（比如斜对角不能都坐人、同列颜色不能相同），就叫兼容。DP 转移只发生在兼容的状态对之间。
- **多进制状态：** 二进制掩码只能表达"选/不选"；染三色要每列 3 种取值，所以用 `itertools.product(range(3), repeat=m)` 枚举一行所有颜色组合，相当于三进制的"行状态"。
- **状态聚合（ABA/ABC 分类）：** 宽度 3 的三色行虽然形式上有很多，但按"第 1 格是否等于“分段实现”部分"可以分成两类：ABA 型（如 1,2,1）和 ABC 型（三格互不相同）。颜色叫什么名字无所谓（重命名对称性），同类状态的转移数目完全一样，所以每类只存一个计数，两个数字就能递推。
- **取模（MOD=10^9+7）：** 计数题答案太大，边加边取模防溢出（Python 里防的是数字变大变慢）。

## 3. 思路推导：从小例子开始

### 3.1 考场排座：行掩码 + 兼容检查

- **Step 1：** 每一行先算出"能坐的座位掩码" allowed：第 i 列是 '.' 就把第 i 位点亮。
- **Step 2：** 枚举这一行所有 2^W 种坐法 mask，保留合法的：一是不能坐坏座位（mask 必须是 allowed 的子集），二是行内不能左右相邻（mask 与 mask 左移一位没有公共位）。
- **Step 3：** 和上一行兼容的条件是斜对角没人：mask 不能和上一行 prev 左移一位、右移一位的结果有公共位（这两次移位分别对应左上、右上方向的冲突）。
- **Step 4：** dp[mask] = 这一行的 mask 人数（bit_count）+ 上一行所有兼容 prev 的 dp 最大值。最后一行处理完取全局最大。

**手算 `['.#.','...']`（“运行示例”部分的表格就是这个）：** 第一行 '.#.' 的 allowed=101₂，合法 mask 有 000（0 人）、001（1 人）、100（1 人）、101（2 人，第 0、2 列不相邻所以合法）。第二行 '...' 的 allowed=111₂，合法 mask 有 000、001、010、100、101（011、110 因为左右相邻被排除）。转移举例：第二行取 101（坐第 0、2 列）时，上一行 prev=101 与它兼容吗？prev 左移、右移分别是 010（溢出舍去）和 1010，和 101 按位与都是 0——因为占据的列是 0 和 2，上一行占据的也是 0 和 2，同列不冲突（本就没有竖直禁坐规则），斜方向恰好错开。于是 dp 值 = 2 + 2 = 4，这就是全局最优。

### 3.2 三色染网格：枚举行状态 + 兼容表

- **Step 1：** 一行有 m 格，每格 3 色，先枚举所有 3^m 种颜色组合，筛掉行内相邻同色的，剩下的就是合法行状态（一共 3·2^(m−1) 个）。
- **Step 2：** 预先算好兼容表：状态 a 和状态 b 兼容 = 每一列的颜色都不同（竖直相邻不同色）。
- **Step 3：** dp 初始为每个合法状态 1 种（第一行）；每加一行，新 dp[b] = 所有兼容 a 的 dp[a] 之和（取模）；n 行之后把 dp 全加起来。

**手算 m=1（宽 1）验证：** 合法行状态就是 3 种单色；兼容 = 不同色。n=1 → 3；n=2 → 3×2=6；n=3 → 3×2×2=12。函数 `color_the_grid(1,1)==3` 正是这个最小例子。

### 3.3 N×3 聚合递推：两个数字搞定

- **Step 1：** 宽 3 的合法行分两类：ABA 型（第 1 格 = “分段实现”部分，如 121）和 ABC 型（三格互不相同，如 123）。单独一行各有 6 个（ABA：3×2=6；ABC：3!=6）。
- **Step 2：** 数一数转移数。以上一行 (1,2,1) 为例：它能接的 ABA 行是每列都不同色且形如 (a,b,a) 的——枚举得 (2,1,2)、(2,3,2)、(3,1,3) 共 **3** 个；能接的 ABC 行是三格互异且逐列不同色的——枚举得 (2,1,3)、(3,1,2) 共 **2** 个。由颜色重命名的对称性，任何 ABA 行都是"3 接 ABA、2 接 ABC"；同理任何 ABC 行是"2 接 ABA、2 接 ABC"。
- **Step 3：** 于是递推：`aba,abc = 3*aba+2*abc, 2*aba+2*abc`。

**手算：** n=1：aba=6，abc=6，合计 12。n=2：aba=3·6+2·6=30，abc=2·6+2·6=24，合计 **54**。这正是 LC1411 的两个官方样例值。

## 4. 核心代码：max_students_profile

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_students_profile(seats):
    if not seats:
        return 0
    width = len(seats[0])
    dp = {0: 0}
    for row in seats:
        allowed = sum((1 << i for i, x in enumerate(row) if x == '.'))
        states = [mask for mask in range(1 << width) if mask & allowed == mask and (not mask & mask << 1)]
        nxt = {}
        for mask in states:
            nxt[mask] = mask.bit_count() + max((v for prev, v in dp.items() if not mask & prev << 1 and (not mask & prev >> 1)))
        dp = nxt
    return max(dp.values())
```

### 逐段读懂代码

### 4.1 `max_students_profile(seats)`

```python
def max_students_profile(seats):
    if not seats:
        return 0
    width = len(seats[0])
    dp = {0: 0}
    for row in seats:
        allowed = sum((1 << i for i, x in enumerate(row) if x == '.'))
        states = [mask for mask in range(1 << width) if mask & allowed == mask and (not mask & mask << 1)]
        nxt = {}
        for mask in states:
            nxt[mask] = mask.bit_count() + max((v for prev, v in dp.items() if not mask & prev << 1 and (not mask & prev >> 1)))
        dp = nxt
    return max(dp.values())
```

- `if not seats: return 0`：空教室没人可坐，挡住空输入。
- `dp={0:0}`：DP 用字典存"上一行的合法掩码 → 最好成绩"。初始是"一行都还没处理"，掩码 0、0 人；它和任何 mask 都兼容（`0&(任何)==0`），所以第一行自然没有上一行约束。
- `allowed=sum(1<<i for i,x in enumerate(row) if x=='.')`：第 i 列能坐就把第 i 位置 1，生成器直接求和拼出掩码。
- `mask&allowed==mask`：判断 mask 是 allowed 的子集——mask 里为 1 的位在 allowed 里也得是 1，等价于"没坐坏座位"。
- `not mask&(mask<<1)`：mask 左移一位后如果还和 mask 有公共 1，说明某两个相邻列同时坐了人；取反表示"没有左右相邻"。
- `nxt[mask]=mask.bit_count()+max(...)`：`bit_count()` 数 mask 里 1 的个数，即这一行坐了几人；`max` 在上一行所有兼容 prev 里挑最好的成绩。兼容条件 `not mask&(prev<<1) and not mask&(prev>>1)` 分别挡住左上/右上斜对角冲突（prev 移位后与 mask 对齐到斜方向的列）。
- `dp=nxt`：换行。字典里只留这一行的合法状态，更早的行被"遗忘"。
- `return max(dp.values())`：最后一行各状态里的最大值就是答案。

### 4.2 `color_the_grid(m, n)`

```python
def color_the_grid(m, n):
    if min(m, n) < 0:
        raise ValueError('nonnegative dimensions required')
    if not m or not n:
        return 1
    states = [p for p in product(range(3), repeat=m) if all((a != b for a, b in zip(p, p[1:])))]
    compatible = [[j for j, b in enumerate(states) if all((x != y for x, y in zip(a, b)))] for a in states]
    dp = [1] * len(states)
    for _ in range(1, n):
        dp = [sum((dp[j] for j in neighbors)) % MOD for neighbors in compatible]
    return sum(dp) % MOD
```

- `if min(m,n)<0: raise ValueError(...)`：负维度非法，直接抛异常。
- `if not m or not n: return 1`：m 或 n 为 0 表示空网格，"什么都不染"恰好 1 种方案。
- `states=[...]`：`product(range(3),repeat=m)` 枚举一行的全部 3^m 种颜色组合；`all(a!=b for a,b in zip(p,p[1:]))` 检查行内相邻两格颜色互不相同。
- `compatible=[[...] for a in states]`：对每个状态 a，收集所有"逐列颜色都不同"的状态下标 j，形成兼容表（相当于邻接表）。
- `dp=[1]*len(states)`：第一行，每个合法状态本身就是 1 种染法。
- `dp=[sum(dp[j] for j in neighbors)%MOD for neighbors in compatible]`：每多一行，新 dp 按兼容表把旧 dp 求和取模；`for neighbors in compatible` 的顺序保证第 b 项用的正是 b 的兼容列表。
- `return sum(dp)%MOD`：n 行结束后，所有末行状态的方案数加起来。

### 4.3 `num_of_ways_n3(n)`

```python
def num_of_ways_n3(n):
    if n == 0:
        return 1
    aba = abc = 6
    for _ in range(1, n):
        aba, abc = ((3 * aba + 2 * abc) % MOD, (2 * aba + 2 * abc) % MOD)
    return (aba + abc) % MOD
```

- `if n==0: return 1`：0 行的网格有 1 种"空染法"，边界约定与 color_the_grid 一致。
- `aba=abc=6`：一行的两类合法状态各 6 个（3.3 节算过）。
- 元组赋值 `aba,abc=(3*aba+2*abc)%MOD,(2*aba+2*abc)%MOD`：右边先整体求值再同时更新，避免先改 aba 影响 abc；系数 3、2、2、2 就是 3.3 节枚举出的转移数。
- `return (aba+abc)%MOD`：两类相加就是总方案数。

## 5. 分段实现：按顺序运行

**本章接口：** `max_students_profile(seats)`、`color_the_grid(m, n)`、`num_of_ways_n3(n)`

### 导入本章使用的库

In [1]:
from itertools import product

### MOD

In [2]:
MOD = 10 ** 9 + 7

### max_students_profile

In [3]:
def max_students_profile(seats):
    if not seats:
        return 0
    width = len(seats[0])
    dp = {0: 0}
    for row in seats:
        allowed = sum((1 << i for i, x in enumerate(row) if x == '.'))
        states = [mask for mask in range(1 << width) if mask & allowed == mask and (not mask & mask << 1)]
        nxt = {}
        for mask in states:
            nxt[mask] = mask.bit_count() + max((v for prev, v in dp.items() if not mask & prev << 1 and (not mask & prev >> 1)))
        dp = nxt
    return max(dp.values())

### color_the_grid

In [4]:
def color_the_grid(m, n):
    if min(m, n) < 0:
        raise ValueError('nonnegative dimensions required')
    if not m or not n:
        return 1
    states = [p for p in product(range(3), repeat=m) if all((a != b for a, b in zip(p, p[1:])))]
    compatible = [[j for j, b in enumerate(states) if all((x != y for x, y in zip(a, b)))] for a in states]
    dp = [1] * len(states)
    for _ in range(1, n):
        dp = [sum((dp[j] for j in neighbors)) % MOD for neighbors in compatible]
    return sum(dp) % MOD

### num_of_ways_n3

In [5]:
def num_of_ways_n3(n):
    if n == 0:
        return 1
    aba = abc = 6
    for _ in range(1, n):
        aba, abc = ((3 * aba + 2 * abc) % MOD, (2 * aba + 2 * abc) % MOD)
    return (aba + abc) % MOD

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

seats=['.#.','...']; width=3; rows=[]
for r,row in enumerate(seats):
    allowed=sum(1<<i for i,c in enumerate(row) if c=='.')
    for mask in range(1<<width):
        if mask&allowed==mask and not mask&(mask<<1): rows.append((r,format(mask,'03b'),mask.bit_count()))
show_table(['行','合法座位掩码','人数'],rows)
print('全局最多人数',max_students_profile(seats))

行,合法座位掩码,人数
0,000,0
0,001,1
0,100,1
0,101,2
1,000,0
1,001,1
1,010,1
1,100,1
1,101,2


全局最多人数 4


## 7. 正确性、适用条件与复杂度

给定上一行状态，未来只受该行边界影响，更早行可被遗忘。合法状态完整枚举了行内选择，兼容关系精确表达跨行约束。宽度3的三色状态按ABA与ABC两类聚合，颜色重命名对称性保证同类转移数相同，得到常数状态递推。

**代价：** 选座保守O(R·4^W)时间、O(2^W)状态空间；三色状态数S=3·2^(m−1)，预计算兼容O(mS²)，后续O(nS²)时间和O(S²)空间；宽3聚合版O(n)时间O(1)空间。指数宽度必须小。

### 展开理解

**为什么可以忘掉更早的行？** 这些题的规则都只"够得着"相邻一行：斜对角冲突只看上一行，竖直颜色约束只看上一行。所以只要上一行的掩码定了，未来能怎么填就完全定了，和再往上的行怎么填的没有关系。这就是"未来只由边界决定"，也是轮廓 DP 这个名字的由来——你只带着"轮廓线"上的信息往前走。

**为什么转移完备？** 合法状态列表把"一行内部的每一种合规选法"都列了出来（行内约束在源头就过滤掉）；兼容条件把"跨行的每一条禁令"精确翻译成位运算/逐列比较。于是任何一种全局合规方案都能被拆成"逐行合法 + 相邻行兼容"的组合，DP 的 max/sum 恰好不重不丢地把它数出来/择优出来。

**ABA/ABC 聚合为什么合法？** 染色约束只关心"同不同色"，不关心"具体是哪种色"。把 (1,2,1) 的颜色整体换名成 (3,1,3)，它能接的后继数量一个不多一个不少。所以同型状态可以共用同一个计数，两类各一个数字就够。

**复杂度（拿具体数字感受）：** 排座是每行 O(4^W)（合法状态最多 2^W 个，每个再扫上一行最多 2^W 个 prev，这是保守估计，实际合法状态少得多）、空间 O(2^W)：LC1349 宽度最多 8，4^8=65536，一行一眨眼，R 行也只要几毫秒。但宽度涨到 20 就是 2^20≈100 万状态、万亿级配对——彻底爆炸，所以"指数必须压在窄的维度上"。三色版合法状态 S=3·2^(m−1)：m=5 时 S=48，兼容表预计算 O(m·S²)≈1.2 万次比较，之后每行 O(S²)=2304，n=1000 行总共约 230 万次操作，瞬间完成。聚合版更是 O(n) 时间、O(1) 空间：n=10^9 也只要 10 亿次加乘……其实题目 n≤5000，几步就出结果。

## 8. 单元测试：每个用例在检查什么

```python
assert color_the_grid(1, 1) == 3 and num_of_ways_n3(1) == 12

assert max_students_profile(['...']) == 2

for n in range(1, 9):
    assert num_of_ways_n3(n) == color_the_grid(3, n)
```

- 第 1 条：两个最小边界。1×1 网格 3 种颜色；宽 3 一行共 12 种染法（6 ABA + 6 ABC）。
- 第 2 条：单行三座、不许左右相邻，最多坐 2 人（第 0、2 列），专测"没有上一行"时 dp={0:0} 的初始转移。
- 第 3 条循环：把聚合递推 `num_of_ways_n3` 和通用 DP `color_the_grid(3,n)` 对拍 8 组——如果 3.3 节那组系数（3、2、2、2）数错了，这里立刻露馅。

第 4 段（排座暴力对拍）：对 2×3 网格枚举全部 2^6=64 种坏座位分布，构造 seats；再枚举全部 2^6 种坐人子集，用 `abs(c-d)==1 and abs(r-s)<=1` 过滤冲突——这个条件覆盖"同一行左右相邻"（行差 0、列差 1）和"斜对角"（行列差都为 1），但**不含正上正下**（列差 0），正好对应考场题"可以坐正前后"的规则。断言 DP 结果等于暴力最大人数。这一段等于把规则的每一个方向都测到了。

第 5 段（染色暴力对拍）：对 (m,n)=(1,3)、(2,2)、(2,3) 三个小网格，穷举全部 3^(m·n) 种染色，逐条检查横向竖向相邻不同色后计数，与 `color_the_grid` 对比。三组尺寸分别覆盖"单列""方阵""双行三列"的形态。

In [7]:
assert color_the_grid(1, 1) == 3 and num_of_ways_n3(1) == 12

assert max_students_profile(['...']) == 2

for n in range(1, 9):
    assert num_of_ways_n3(n) == color_the_grid(3, n)

for broken in range(1 << 6):
    seats = [''.join(('#' if broken >> r * 3 + c & 1 else '.' for c in range(3))) for r in range(2)]
    valid = []
    for mask in range(1 << 6):
        if mask & broken:
            continue
        cells = [(i // 3, i % 3) for i in range(6) if mask >> i & 1]
        if all((not (abs(c - d) == 1 and abs(r - s) <= 1) for i, (r, c) in enumerate(cells) for s, d in cells[i + 1:])):
            valid.append(mask.bit_count())
    assert max_students_profile(seats) == max(valid)

for m, n in [(1, 3), (2, 2), (2, 3)]:
    count = 0
    for colors in product(range(3), repeat=m * n):
        if all((colors[r * n + c] != colors[(r + 1) * n + c] for r in range(m - 1) for c in range(n))) and all((colors[r * n + c] != colors[r * n + c + 1] for r in range(m) for c in range(n - 1))):
            count += 1
    assert color_the_grid(m, n) == count

print('N116: 所有本章断言通过')

N116: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 交换长宽以减少状态数但核对规则对称性。

**练习 2：** 与二分匹配解考场题对照。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（交换长宽）：** 提示——轮廓 DP 的指数在宽度上，所以永远让窄的那一边当"宽度"：10×3 的网格应当转置成 3×10 再跑。但转置前必须核对规则在转置下是否不变：排座题的"左右 + 斜对角"约束转置后仍是"左右 + 斜对角"（对称），可以换；可如果某题规则是"只禁左邻不禁上邻"，转置就改变题意了。建议拿 2×3 与 3×2 的座位表各手算一遍，确认答案一致，再用暴力对拍。
- **练习 2（与二分匹配对照）：** 提示——把每个能坐的座位看成点，有冲突（横邻/斜对角）的座位之间连边，"最多坐多少人"就是这张冲突图上的最大独立集。试着把座位按**列的奇偶**分到两侧，你会发现横邻列差 1、斜对角列差也是 1，冲突边必然跨两侧——冲突图是二分图，于是可以用"点数 − 最大匹配"求最大独立集。先在小网格（比如 2×3）上手算匹配数对照 DP 答案，再讨论两种解法各自的适用范围（匹配解不依赖宽度小，但建图代价大）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from itertools import product

MOD = 10 ** 9 + 7

def max_students_profile(seats):
    if not seats:
        return 0
    width = len(seats[0])
    dp = {0: 0}
    for row in seats:
        allowed = sum((1 << i for i, x in enumerate(row) if x == '.'))
        states = [mask for mask in range(1 << width) if mask & allowed == mask and (not mask & mask << 1)]
        nxt = {}
        for mask in states:
            nxt[mask] = mask.bit_count() + max((v for prev, v in dp.items() if not mask & prev << 1 and (not mask & prev >> 1)))
        dp = nxt
    return max(dp.values())

def color_the_grid(m, n):
    if min(m, n) < 0:
        raise ValueError('nonnegative dimensions required')
    if not m or not n:
        return 1
    states = [p for p in product(range(3), repeat=m) if all((a != b for a, b in zip(p, p[1:])))]
    compatible = [[j for j, b in enumerate(states) if all((x != y for x, y in zip(a, b)))] for a in states]
    dp = [1] * len(states)
    for _ in range(1, n):
        dp = [sum((dp[j] for j in neighbors)) % MOD for neighbors in compatible]
    return sum(dp) % MOD

def num_of_ways_n3(n):
    if n == 0:
        return 1
    aba = abc = 6
    for _ in range(1, n):
        aba, abc = ((3 * aba + 2 * abc) % MOD, (2 * aba + 2 * abc) % MOD)
    return (aba + abc) % MOD

# 示例与回归测试
assert color_the_grid(1, 1) == 3 and num_of_ways_n3(1) == 12

assert max_students_profile(['...']) == 2

for n in range(1, 9):
    assert num_of_ways_n3(n) == color_the_grid(3, n)

for broken in range(1 << 6):
    seats = [''.join(('#' if broken >> r * 3 + c & 1 else '.' for c in range(3))) for r in range(2)]
    valid = []
    for mask in range(1 << 6):
        if mask & broken:
            continue
        cells = [(i // 3, i % 3) for i in range(6) if mask >> i & 1]
        if all((not (abs(c - d) == 1 and abs(r - s) <= 1) for i, (r, c) in enumerate(cells) for s, d in cells[i + 1:])):
            valid.append(mask.bit_count())
    assert max_students_profile(seats) == max(valid)

for m, n in [(1, 3), (2, 2), (2, 3)]:
    count = 0
    for colors in product(range(3), repeat=m * n):
        if all((colors[r * n + c] != colors[(r + 1) * n + c] for r in range(m - 1) for c in range(n))) and all((colors[r * n + c] != colors[r * n + c + 1] for r in range(m) for c in range(n - 1))):
            count += 1
    assert color_the_grid(m, n) == count

print('N116: 所有本章断言通过')

N116: 所有本章断言通过


## 11. 代表题与迁移

- **1349. Maximum Students Taking Exam**：这就是 `max_students_profile` 的原题——练"二进制行掩码 + 行内预过滤 + 斜对角兼容检查"的完整流程。
- **1931. Painting a Grid With Three Different Colors**：这就是 `color_the_grid` 的原题——练"多进制行状态 + 预计算兼容表 + 逐行求和取模"的计数型轮廓 DP。
- **1411. Number of Ways to Paint N × 3 Grid**：这就是 `num_of_ways_n3` 的原题——练"利用对称性把状态聚合成常数个类别"的降维技巧，是 1931 在固定宽度下的快解。

**题面入口：** [1349](https://leetcode.com/problems/maximum-students-taking-exam/) · [1411](https://leetcode.com/problems/number-of-ways-to-paint-n-3-grid/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。